In [10]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- 1. Load Tokenizer and Label Map ---
with open(r"C:\Users\jange\Downloads\tokenizer.pkl", 'rb') as f:
    tokenizer = pickle.load(f)

with open(r"C:\Users\jange\Downloads\label_map.pkl", 'rb') as f:
    label_map = pickle.load(f)

# Reverse label map for decoding
inv_label_map = {v: k for k, v in label_map.items()}

# --- 2. Load Trained Model ---
model = load_model(r"C:\Users\jange\Downloads\lstm_sentiment_model.h5")

# --- 3. Define Prediction Function ---
def predict_sentiment(texts, max_len=100):
    if isinstance(texts, str):
        texts = [texts]

    # Preprocess: tokenize + pad
    seqs = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(seqs, maxlen=max_len, padding='post')

    # Predict
    probs = model.predict(padded)
    if probs.shape[1] == 1:  # binary
        preds = (probs > 0.5).astype(int).flatten()
    else:  # multi-class
        preds = np.argmax(probs, axis=1)

    # Map back to labels
    labels = [inv_label_map[p] for p in preds]
    return labels

# --- 4. Try Custom Prediction ---
tweets = [
    "Canada is doing amazing in the World Cup!",
    "The new iPhone is just a cash grab.",
    "Not sure how I feel about this movie tbh.",
    "I had a terrible experience at that restaurant.",
    "Elon Musk is wilding again 😂"                                                        # sarcastic/negative
]

results = predict_sentiment(tweets)
for txt, label in zip(tweets, results):
    print(f"🗨️ \"{txt}\" → 🏷️ {label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
🗨️ "Canada is doing amazing in the World Cup!" → 🏷️ positive
🗨️ "The new iPhone is just a cash grab." → 🏷️ neutral
🗨️ "Not sure how I feel about this movie tbh." → 🏷️ negative
🗨️ "I had a terrible experience at that restaurant." → 🏷️ negative
🗨️ "Elon Musk is wilding again 😂" → 🏷️ neutral
